In [8]:
import pandas as pd
import numpy as np

path = "../data/raw/home-credit-default-risk/"

# main table
app = pd.read_csv(path + "application_train.csv")

# bureau, one row per applicant
bureau = pd.read_csv(path + "bureau.csv")
bureau["ACTIVE"] = (bureau["CREDIT_ACTIVE"] == "Active").astype(int)
bureau_agg = bureau.groupby("SK_ID_CURR").agg(
    bureau_count=("SK_ID_BUREAU", "count"),
    bureau_active=("ACTIVE", "sum"),
    bureau_credit_sum=("AMT_CREDIT_SUM", "sum"),
    bureau_debt_sum=("AMT_CREDIT_SUM_DEBT", "sum"),
    bureau_max_overdue=("CREDIT_DAY_OVERDUE", "max")
).reset_index()

# previous applications, one row per applicant
prev = pd.read_csv(path + "previous_application.csv")
prev["REFUSED"] = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype(int)
prev_agg = prev.groupby("SK_ID_CURR").agg(
    prev_count=("SK_ID_PREV", "count"),
    prev_refused=("REFUSED", "sum"),
    prev_refused_rate=("REFUSED", "mean"),
    prev_avg_credit=("AMT_CREDIT", "mean")
).reset_index()

print(app.shape, bureau_agg.shape, prev_agg.shape)

(307511, 122) (305811, 6) (338857, 5)


In [10]:
df = app.merge(bureau_agg, on="SK_ID_CURR", how="left")
df = df.merge(prev_agg, on="SK_ID_CURR", how="left")

# the merge should NOT change the number of rows
print(df.shape)
print(len(df) == len(app))

(307511, 131)
True


In [11]:
# things we found in the first notebook
df["AGE"] = -df["DAYS_BIRTH"] / 365
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

# ratio features
df["CREDIT_INCOME_RATIO"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
df["ANNUITY_INCOME_RATIO"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]

# people with no bureau/previous history have NaN in the counts, 0 makes more sense
count_cols = ["bureau_count", "bureau_active", "prev_count", "prev_refused"]
df[count_cols] = df[count_cols].fillna(0)

In [12]:
print(df[["TARGET", "CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "prev_refused_rate"]].corr()["TARGET"])

TARGET                  1.000000
CREDIT_INCOME_RATIO    -0.007727
ANNUITY_INCOME_RATIO    0.014265
prev_refused_rate       0.077671
Name: TARGET, dtype: float64


In [13]:
df.to_csv("../data/processed/train_merged.csv", index=False)
print(df.shape)

(307511, 134)
